# 📊 PHÂN CỤM SINH VIÊN LỚP K58KTP – K-MEANS

> **Môn học:** Học máy / Phân tích dữ liệu  
> **Phương pháp:** K-Means Clustering (K=3)  
> **Dữ liệu:** Bảng điểm toàn khóa lớp K58KTP  

---

## 📦 Bước 1: Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
import warnings
warnings.filterwarnings('ignore')

print('✅ Import thư viện thành công!')

## 📂 Bước 2: Đọc dữ liệu từ file Excel

In [ ]:
FILE = 'TỔNG_HỢP_ĐIỂM_K58KTP.xlsx'  # 📌 Đặt file xlsx cùng thư mục với notebook này

df = pd.read_excel(FILE, header=None)

mssv_list    = df.iloc[1, 3:].tolist()
name_list    = df.iloc[2, 3:].tolist()
subject_list = df.iloc[4:, 2].tolist()

print(f'📋 Số sinh viên  : {len(mssv_list)}')
print(f'📚 Số môn học    : {len(subject_list)}')
print(f'\n👤 5 sinh viên đầu:')
for i in range(5):
    print(f'   {mssv_list[i]}  –  {name_list[i]}')

## 🧹 Bước 3: Làm sạch & tính GPA trung bình

In [ ]:
def safe_num(x):
    try:
        v = float(x)
        return v if 0 <= v <= 4 else np.nan
    except:
        return np.nan

scores_raw   = df.iloc[4:, 3:].copy()
scores_clean = scores_raw.map(safe_num)
scores_clean.columns = range(len(mssv_list))
scores_clean.index   = range(len(scores_clean))

student_df = scores_clean.T
student_df.index   = mssv_list
student_df.columns = subject_list
student_df['GPA_TB'] = student_df.mean(axis=1)
student_df['Name']   = name_list

# Lọc bỏ SV chưa nhập điểm
before = len(student_df)
student_df = student_df.dropna(subset=['GPA_TB'])
after  = len(student_df)

print(f'👥 Tổng ban đầu  : {before} sinh viên')
print(f'🗑  Đã loại bỏ   : {before - after} SV (chưa nhập điểm)')
print(f'✅ Còn lại       : {after} sinh viên')
print(f'\n📊 GPA trung bình lớp: {student_df["GPA_TB"].mean():.3f}')
print(f'📈 GPA cao nhất      : {student_df["GPA_TB"].max():.3f}')
print(f'📉 GPA thấp nhất     : {student_df["GPA_TB"].min():.3f}')

## ⚙️ Bước 4: Tiền xử lý – Impute & Chuẩn hóa

In [ ]:
X = student_df.drop(columns=['GPA_TB', 'Name'])

# Điền NaN bằng trung bình cột
imputer  = SimpleImputer(strategy='mean')
X_imp    = imputer.fit_transform(X)

# Chuẩn hóa về mean=0, std=1
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)

print(f'📐 Ma trận đặc trưng sau xử lý: {X_scaled.shape}')
print(f'   → {X_scaled.shape[0]} sinh viên  ×  {X_scaled.shape[1]} môn học')

## 📐 Bước 5: Elbow Method – Tìm K tối ưu

In [ ]:
inertias = []
K_range  = range(1, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

# Vẽ biểu đồ Elbow
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(K_range), inertias, 'o-', color='#1A3C6E', linewidth=2.5, markersize=8)
ax.axvline(x=3, color='#E74C3C', linestyle='--', linewidth=2, label='K = 3 (chọn)')
ax.fill_between(list(K_range), inertias, alpha=0.08, color='#1A3C6E')
for k, v in zip(K_range, inertias):
    ax.annotate(f'{v:.0f}', (k, v), textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=8, color='#444')
ax.set_title('Elbow Method – Chọn số cụm K tối ưu', fontsize=13, fontweight='bold')
ax.set_xlabel('Số cụm K'); ax.set_ylabel('Inertia (SSE)')
ax.legend(fontsize=11); ax.grid(alpha=0.3)
ax.set_facecolor('#FAFAFA')
plt.tight_layout()
plt.show()
print('→ Điểm gãy (elbow) rõ nhất tại K=3 ✔')

## 🤖 Bước 6: Phân cụm K-Means với K=3

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_scaled)

student_df['Cluster'] = labels

# Gán tên nhóm dựa trên GPA trung bình
cluster_gpa = student_df.groupby('Cluster')['GPA_TB'].mean().sort_values(ascending=False)
rank_map = {
    cluster_gpa.index[0]: 'Nhóm 1 - Giỏi',
    cluster_gpa.index[1]: 'Nhóm 2 - Khá',
    cluster_gpa.index[2]: 'Nhóm 3 - Trung bình'
}
student_df['Nhom'] = student_df['Cluster'].map(rank_map)

print(f'✅ Phân cụm hoàn tất! Inertia = {kmeans.inertia_:.2f}\n')
line = '─' * 58
header = '{:<28} {:>6}  {:>8}  {:>6}  {:>6}'.format('Nhóm', 'Số SV', 'GPA TB', 'Min', 'Max')
print(line)
print(header)
print(line)
for nhom in ['Nhóm 1 - Giỏi', 'Nhóm 2 - Khá', 'Nhóm 3 - Trung bình']:
    grp = student_df[student_df['Nhom'] == nhom]['GPA_TB']
    print(f'{nhom:<28} {len(grp):>6}  {grp.mean():>8.3f}  {grp.min():>6.3f}  {grp.max():>6.3f}')
print(line)

## 📋 Bước 7: Xem danh sách từng nhóm

In [ ]:
sep = '=' * 50
for nhom in ['Nhóm 1 - Giỏi', 'Nhóm 2 - Khá', 'Nhóm 3 - Trung bình']:
    grp = student_df[student_df['Nhom'] == nhom][['Name','GPA_TB']].sort_values('GPA_TB', ascending=False).reset_index()
    grp.columns = ['MSSV','Ho va ten','GPA TB']
    grp['GPA TB'] = grp['GPA TB'].round(3)
    grp.index = grp.index + 1
    print('\n' + sep)
    print('  ' + nhom + '  (' + str(len(grp)) + ' sinh vien)')
    print(sep)
    display(grp)

## 📈 Bước 8: Trực quan hóa kết quả (5 biểu đồ)

In [ ]:
# PCA giảm chiều để vẽ scatter
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

colors = {
    'Nhóm 1 - Giỏi'       : '#2196F3',
    'Nhóm 2 - Khá'        : '#4CAF50',
    'Nhóm 3 - Trung bình' : '#FF9800'
}

fig = plt.figure(figsize=(18, 12), facecolor='#F8F9FA')
fig.suptitle('KẾT QUẢ PHÂN CỤM LỚP K58KTP – K-MEANS (K=3)',
             fontsize=15, fontweight='bold', color='#1A3C6E', y=0.99)
gs = GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

# ① Elbow
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(list(K_range), inertias, 'o-', color='#1A3C6E', linewidth=2, markersize=7)
ax1.axvline(x=3, color='#E74C3C', linestyle='--', alpha=0.8, label='K=3')
ax1.fill_between(list(K_range), inertias, alpha=0.1, color='#1A3C6E')
ax1.set_title('① Elbow Method', fontsize=11, fontweight='bold')
ax1.set_xlabel('Số cụm K'); ax1.set_ylabel('Inertia')
ax1.legend(); ax1.grid(alpha=0.3); ax1.set_facecolor('#FAFAFA')

# ② PCA Scatter
ax2 = fig.add_subplot(gs[0, 1:])
for nhom, color in colors.items():
    pos = [i for i, n in enumerate(student_df['Nhom']) if n == nhom]
    ax2.scatter(X_pca[pos, 0], X_pca[pos, 1], c=color,
                label=nhom, s=90, alpha=0.85, edgecolors='white', linewidth=0.8)
centers_pca = pca.transform(kmeans.cluster_centers_)
for i, (cx, cy) in enumerate(centers_pca):
    ax2.scatter(cx, cy, marker='*', s=400, c=colors[rank_map[i]],
                edgecolors='black', linewidth=1.2, zorder=5)
ax2.set_title('② Phân cụm theo PCA (2D)', fontsize=11, fontweight='bold')
ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3); ax2.set_facecolor('#FAFAFA')

# ③ Boxplot
ax3 = fig.add_subplot(gs[1, 0])
data_bp = [student_df[student_df['Nhom']==n]['GPA_TB'].dropna().values
           for n in ['Nhóm 1 - Giỏi','Nhóm 2 - Khá','Nhóm 3 - Trung bình']]
bp = ax3.boxplot(data_bp, patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], ['#2196F3','#4CAF50','#FF9800']):
    patch.set_facecolor(color); patch.set_alpha(0.75)
for med in bp['medians']:
    med.set_color('black'); med.set_linewidth(2)
ax3.set_xticks([1,2,3]); ax3.set_xticklabels(['Giỏi','Khá','TB'])
ax3.set_title('③ Phân phối GPA', fontsize=11, fontweight='bold')
ax3.set_ylabel('GPA trung bình'); ax3.grid(alpha=0.3, axis='y')
ax3.set_facecolor('#FAFAFA')

# ④ Bar số SV
ax4 = fig.add_subplot(gs[1, 1])
counts = [len(student_df[student_df['Nhom']==n]) for n in
          ['Nhóm 1 - Giỏi','Nhóm 2 - Khá','Nhóm 3 - Trung bình']]
bars = ax4.bar(['Giỏi','Khá','TB'], counts,
               color=['#2196F3','#4CAF50','#FF9800'], alpha=0.85, edgecolor='white', linewidth=1.5)
for bar, cnt in zip(bars, counts):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{cnt} SV\n({cnt/after*100:.1f}%)', ha='center', fontsize=10, fontweight='bold')
ax4.set_title('④ Số SV mỗi nhóm', fontsize=11, fontweight='bold')
ax4.set_ylabel('Số sinh viên'); ax4.set_ylim(0, max(counts)*1.3)
ax4.grid(alpha=0.3, axis='y'); ax4.set_facecolor('#FAFAFA')

# ⑤ GPA TB từng nhóm
ax5 = fig.add_subplot(gs[1, 2])
gpa_means = [student_df[student_df['Nhom']==n]['GPA_TB'].mean()
             for n in ['Nhóm 1 - Giỏi','Nhóm 2 - Khá','Nhóm 3 - Trung bình']]
bars2 = ax5.barh(['Giỏi','Khá','TB'], gpa_means,
                 color=['#2196F3','#4CAF50','#FF9800'], alpha=0.85, height=0.5)
for bar, val in zip(bars2, gpa_means):
    ax5.text(val + 0.03, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontweight='bold', fontsize=11)
ax5.set_xlim(0, 4)
ax5.axvline(x=2.0, color='gray', linestyle=':', alpha=0.5)
ax5.axvline(x=3.0, color='gray', linestyle=':', alpha=0.5)
ax5.set_title('⑤ GPA TB mỗi nhóm', fontsize=11, fontweight='bold')
ax5.set_xlabel('GPA (thang 4)'); ax5.grid(alpha=0.3, axis='x')
ax5.set_facecolor('#FAFAFA')

plt.savefig('PhanCum_K58KTP_BieuDo.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Đã lưu biểu đồ: PhanCum_K58KTP_BieuDo.png')

## 💾 Bước 9: Xuất kết quả ra file Excel

In [ ]:
result = student_df[['Name','GPA_TB','Nhom']].copy().reset_index()
result.columns = ['MSSV','Ho_ten','GPA','Nhom']
result = result.sort_values(['Nhom','GPA'], ascending=[True,False]).reset_index(drop=True)
result.insert(0,'STT', range(1, len(result)+1))

wb  = openpyxl.Workbook()
ws  = wb.active
ws.title = 'Kết quả phân cụm'

HDR  = PatternFill('solid', fgColor='1A3C6E')
LB   = PatternFill('solid', fgColor='D6E4F7')
LG   = PatternFill('solid', fgColor='D9F2E6')
LO   = PatternFill('solid', fgColor='FDE8D8')
thin = Side(style='thin', color='BBBBBB')
brd  = Border(left=thin, right=thin, top=thin, bottom=thin)

ws.merge_cells('A1:F1')
ws['A1'] = 'BẢNG TỔNG HỢP ĐIỂM & PHÂN CỤM LỚP K58KTP'
ws['A1'].font = Font(bold=True, size=14, color='FFFFFF')
ws['A1'].fill = HDR
ws['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws.row_dimensions[1].height = 30

ws.merge_cells('A2:F2')
ws['A2'] = f'K-Means (K=3) | {after} sinh viên | Đã loại {before-after} SV chưa nhập điểm'
ws['A2'].font = Font(italic=True, size=10, color='666666')
ws['A2'].alignment = Alignment(horizontal='center')

for col, h in enumerate(['STT','MSSV','Họ và tên','GPA trung bình','Phân nhóm','Xếp loại'], 1):
    c = ws.cell(row=3, column=col, value=h)
    c.font = Font(bold=True, color='FFFFFF')
    c.fill = HDR
    c.alignment = Alignment(horizontal='center', vertical='center')
    c.border = brd
ws.row_dimensions[3].height = 22

fill_map = {'Nhóm 1 - Giỏi': LB, 'Nhóm 2 - Khá': LG, 'Nhóm 3 - Trung bình': LO}
xep_loai = {'Nhóm 1 - Giỏi': '⭐ Xuất sắc / Giỏi', 'Nhóm 2 - Khá': '✅ Khá', 'Nhóm 3 - Trung bình': '📌 Trung bình'}

for _, row in result.iterrows():
    r    = int(row['STT']) + 3
    nhom = row['Nhom']
    gpa  = round(row['GPA'], 3) if pd.notna(row['GPA']) else 'N/A'
    for col, val in enumerate([row['STT'], row['MSSV'], row['Ho_ten'], gpa, nhom, xep_loai.get(nhom,'')], 1):
        cell = ws.cell(row=r, column=col, value=val)
        cell.fill = fill_map.get(nhom, PatternFill())
        cell.border = brd
        cell.alignment = Alignment(horizontal='left' if col==3 else 'center', vertical='center')
    ws.row_dimensions[r].height = 18

for i, w in enumerate([5,18,28,16,24,22], 1):
    ws.column_dimensions[openpyxl.utils.get_column_letter(i)].width = w

out_xlsx = 'KetQua_PhanCum_K58KTP.xlsx'
wb.save(out_xlsx)
print(f'✅ Đã lưu Excel: {out_xlsx}')

---
## 🏁 Tóm tắt kết quả

In [ ]:
print('=' * 55)
print('  KẾT QUẢ PHÂN CỤM LỚP K58KTP')
print('=' * 55)
print(f'  Tổng sinh viên hợp lệ : {after}')
print(f'  Thuật toán             : K-Means (K=3)')
print()
for nhom in ['Nhóm 1 - Giỏi', 'Nhóm 2 - Khá', 'Nhóm 3 - Trung bình']:
    grp = student_df[student_df['Nhom'] == nhom]['GPA_TB']
    print(f'  {nhom}: {len(grp)} SV  |  GPA TB = {grp.mean():.3f}')
print()
top_name = student_df.loc[student_df['GPA_TB'].idxmax(), 'Name']
top_gpa  = student_df['GPA_TB'].max()
print(f'  Top 1: {top_name}  -  GPA {top_gpa:.3f}')
print('=' * 55)
print('  ✔ Excel  : KetQua_PhanCum_K58KTP.xlsx')
print('  ✔ Biểu đồ: PhanCum_K58KTP_BieuDo.png')
print('=' * 55)